In [1]:
# 7_label_clusters.ipynb
#
# For each cluster produced by 6_cluster.ipynb, calls the OpenAI Chat API
# to generate a short title and 2-3 sentence description per cluster,
# then saves enriched results to data/6_cluster/<name>_described.csv.
#
# All clusters for a single LA (Employed 1, Employed 2, Retired 1, …) are
# sent in ONE API call — ~33 calls for London instead of ~130.
#
# Requires OPENAI_API_KEY in environment or .env file.

import sys, os, time, json, shutil
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
from pathlib import Path
from tqdm import tqdm

import importlib
from data_pipeline.config_paths     import DATA_FOLDER
import data_pipeline.config_variables as _cv
import data_pipeline.config_cluster   as _cc
importlib.reload(_cv)
importlib.reload(_cc)

from data_pipeline.config_variables import VARIABLE_MAP
from data_pipeline.config_cluster   import WAVE
from openai import OpenAI

# ── Load .env if present ──────────────────────────────────────────────────────
try:
    from dotenv import load_dotenv
    load_dotenv(Path('..') / '.env', override=False)
    print("Loaded .env")
except ImportError:
    pass

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")
if not OPENAI_API_KEY:
    raise EnvironmentError(
        "OPENAI_API_KEY not set. Export it in your shell or add it to .env:\n"
        "  export OPENAI_API_KEY=sk-..."
    )

# ── Config ────────────────────────────────────────────────────────────────────
MODEL          = "gpt-4.1-mini"   # 32 768 output-token limit; cheap & fast
MAX_OUT_TOKENS = 32_000           # hard cap, safely under the model limit
CLUSTER_CSV = Path(f"../{DATA_FOLDER}/6_cluster/LA_london_clusters.csv")
OUTPUT_CSV  = CLUSTER_CSV.parent / (CLUSTER_CSV.stem + "_described.csv")
MAX_RETRIES = 3
RETRY_DELAY = 5

client = OpenAI(api_key=OPENAI_API_KEY)
print(f"Model:       {MODEL}")
print(f"Cluster CSV: {CLUSTER_CSV}")
print(f"Output CSV:  {OUTPUT_CSV}")

# ── Load CSV ──────────────────────────────────────────────────────────────────
df = pd.read_csv(CLUSTER_CSV)
META_COLS = {'tribe_label', 'size', 'unit_id', 'la_name', 'cluster_level', 'group'}
print(f"Loaded {len(df)} cluster rows across {df['unit_id'].nunique()} units")

# ── Prompts ───────────────────────────────────────────────────────────────────
SYSTEM_PROMPT = """You are a social researcher specialising in UK population demographics.
You will be given the statistical profiles of several population clusters from a single
Local Authority — spanning multiple employment groups (Employed, Retired, Student, etc.) —
derived from the UK Household Longitudinal Study (UKHLS).

Respond with a JSON object with a single key "clusters" whose value is an array.
Each array element corresponds to one input cluster and must contain exactly:
  "tribe_label"  — copied verbatim from the input (used to match results back)
  "title"        — a vivid, memorable 3-5 word name for this persona group
  "description"  — 2-3 sentences describing the defining characteristics in plain English

Notes on the data:
- "Highest qualification": 1=Degree … 5=Other/None. Lower = higher qualification.
- "Social class (NS-SEC 8)": 1=Higher managerial … 8=Routine. Lower = higher class.
- "Mental/Physical health score (SF-12)": 0-100; higher = better health.
- Neighbourhood Cohesion Index 1-5; higher = stronger community.
- Local services ratings 1-5; higher = better satisfaction.
Respond with valid JSON only — no markdown, no explanation outside the JSON."""


def build_la_prompt(unit_id: str, la_name: str, rows: pd.DataFrame) -> str:
    """Build a single prompt containing all clusters for one LA."""
    lines = [
        f"Local Authority: {la_name} ({unit_id})",
        f"Clusters to label: {len(rows)}",
        "",
    ]
    for _, row in rows.iterrows():
        lines += [
            f"--- tribe_label: {row['tribe_label']} ---",
            f"  Employment group: {row.get('group', 'Unknown')}",
            f"  Population size:  {int(row['size']):,}",
        ]
        for col in [c for c in row.index if c not in META_COLS]:
            val = row[col]
            if pd.isna(val):
                continue
            val_str = str(int(val)) if isinstance(val, float) and val == int(val) else (
                f"{val:.1f}" if isinstance(val, float) else str(val)
            )
            lines.append(f"  {col}: {val_str}")
        lines.append("")
    return "\n".join(lines)


# ── One API call per LA ───────────────────────────────────────────────────────
# (unit_id, tribe_label) -> {"gpt_title": ..., "gpt_description": ...}
label_map: dict[tuple, dict] = {}

la_name_col = "la_name" if "la_name" in df.columns else "unit_id"
unit_ids = df["unit_id"].unique().tolist()

for unit_id in tqdm(unit_ids, desc="Labelling LAs"):
    la_rows = df[df["unit_id"] == unit_id]
    la_name = la_rows[la_name_col].iloc[0]
    prompt  = build_la_prompt(unit_id, la_name, la_rows)

    response_text = None
    for attempt in range(MAX_RETRIES):
        try:
            resp = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user",   "content": prompt},
                ],
                response_format={"type": "json_object"},
                temperature=0.7,
                max_tokens=min(200 * len(la_rows) + 100, MAX_OUT_TOKENS),
            )
            response_text = resp.choices[0].message.content
            break
        except Exception as e:
            wait = RETRY_DELAY * (2 ** attempt)
            print(f"\n  Error on {unit_id}: {e}  — retrying in {wait}s")
            time.sleep(wait)

    if not response_text:
        continue

    try:
        parsed = json.loads(response_text)
        # Expect {"clusters": [...]} but tolerate a bare list or other wrapper key
        items = parsed.get("clusters") or next(
            (v for v in parsed.values() if isinstance(v, list)), []
        )
        for item in items:
            key = (unit_id, item.get("tribe_label", ""))
            label_map[key] = {
                "gpt_title":       (item.get("title", "") or "").strip() or None,
                "gpt_description": (item.get("description", "") or "").strip() or None,
            }
    except (json.JSONDecodeError, AttributeError) as exc:
        print(f"\n  Parse error for {unit_id}: {exc}")

successful = sum(1 for v in label_map.values() if v["gpt_title"])
print(f"\nCompleted {len(unit_ids)} LA calls — {successful}/{len(df)} clusters labelled")

# ── Merge back and save ───────────────────────────────────────────────────────
df["gpt_title"]       = df.apply(lambda r: label_map.get((r["unit_id"], r["tribe_label"]), {}).get("gpt_title"),       axis=1)
df["gpt_description"] = df.apply(lambda r: label_map.get((r["unit_id"], r["tribe_label"]), {}).get("gpt_description"), axis=1)

df.to_csv(OUTPUT_CSV, index=False)
print(f"Saved {len(df)} rows to {OUTPUT_CSV}")
display(df[["unit_id", "group", "tribe_label", "size", "gpt_title", "gpt_description"]].head(12))

# ── Copy to api/ for deployment ───────────────────────────────────────────────
API_CLUSTERS_DIR = Path('..') / 'api' / 'data' / 'clusters'
API_CLUSTERS_DIR.mkdir(parents=True, exist_ok=True)
shutil.copy(OUTPUT_CSV, API_CLUSTERS_DIR / OUTPUT_CSV.name)
print(f'Copied {OUTPUT_CSV.name}  ->  api/data/clusters/')


Model:       gpt-4.1-mini
Cluster CSV: ../data/6_cluster/LA_london_clusters.csv
Output CSV:  ../data/6_cluster/LA_london_clusters_described.csv
Loaded 480 cluster rows across 12 units


Labelling LAs: 100%|██████████| 12/12 [03:14<00:00, 16.19s/it]


Completed 12 LA calls — 120/480 clusters labelled
Saved 480 rows to ../data/6_cluster/LA_london_clusters_described.csv


,unit_id,group,tribe_label,size,gpt_title,gpt_description
0,E09000001,Employed,Employed 1,1608,Family-Focused Mid-Career Men,Predominantly male workers around 40 years old...
1,E09000001,Employed,Employed 2,4208,Young Urban Professionals,Mostly men in their early 40s without children...
2,E09000001,Retired,Retired 1,1610,Older Christian Retirees,"Older retirees, mostly women in their late 70s..."
3,E09000001,Retired,Retired 2,403,Younger Independent Retired Men,Predominantly men in their late 60s with good ...
4,E09000001,Unemployed,Unemployed 1,212,Middle-Aged Unemployed with Families,Middle-aged men experiencing unemployment with...
5,E09000001,Unemployed,Unemployed 2,253,Young Multilingual Job Seekers,"Young men, all with English as a second langua..."
6,E09000001,Student,Student,334,Young Daily Commuting Students,"Young adults around 24, mostly male students w..."
7,E09000001,On leave,On leave,225,Predominantly Female On-Leave Parents,Mostly women around 40 years old on leave from...
8,E09000001,Inactive,Inactive,406,Middle-Aged Inactive Men with Low Wellbeing,Men in their late 40s who are inactive in the ...
9,Employed: 100%,LA,Employed 1,37776,Diverse Midlife Workforce,This large group of employed individuals is pr...


Copied LA_london_clusters_described.csv  ->  api/data/clusters/
